<a href="https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/AI%E6%8F%90%E5%95%8F%E8%88%87%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92%E5%AF%A6%E6%88%B0/06_商場客群輪廓探索_進階分群篇/商場客群輪廓探索_進階分群篇.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [ ]:
%%capture
# 在 Google Colab 執行時，先安裝中文字型與資料下載支援
%pip install wget mplfonts scipy seaborn


# 🧭 範例 06：商場客群輪廓探索（進階分群篇） — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本筆記本沿用範例 01 的 **「一段程式碼 + 一段 AI 互動學習指引」** 架構：

- 每個程式儲存格都提供可直接執行的資料處理、建模與評估流程。
- 每段程式碼的正下方都有白話說明與專屬 Prompt；可直接複製給 AI 追問原理或練習調整。
- 資料取自《資料科學自學聖經》的 `customer.csv`，在 Colab 找不到檔案時會自動下載。

請從步驟 0 依序執行。交叉驗證與調參只能使用訓練資料；測試集保留到最後一次驗收。


---  
### 【步驟 0】：環境準備與繁體中文字型設定
載入資料科學套件並設定中文字型，避免圖表顯示亂碼。


In [ ]:
# [步驟 0 程式碼] 載入套件與設定繁體中文字型
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK TC")
except Exception:
    print("提示：中文字型設定失敗時，模型仍可正常執行。")
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid")


#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

載入資料處理與視覺化套件後，先確認環境再開始模型實驗；中文字型失敗不會影響模型訓練。

> ❓ 請用生活化比喻解釋這一步的原理，並指出一個常見誤解。

> 🛠️ 請只改一個超參數，比較修改前後的結果；不要使用測試集選參數。


---  
### 【步驟 1】：資料載入與結構探索（EDA）
讀取資料、查看欄位型態與摘要統計，先了解資料再下模型。


In [ ]:
# [步驟 1 程式碼] 資料載入與結構探索
if not os.path.exists("customer.csv"):
    import wget
    wget.download("https://raw.githubusercontent.com/roberthsu2003/machine_learning/main/AI%E6%8F%90%E5%95%8F%E8%88%87%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92%E5%AF%A6%E6%88%B0/06_商場客群輪廓探索_進階分群篇/customer.csv", "customer.csv")
df = pd.read_csv("customer.csv")
print(f"資料筆數：{len(df):,}；欄位數：{df.shape[1]}")
display(df.head())
display(df.describe(include="all").T)
print("非監督式分群；沒有標籤，目標是探索客群結構")


#### 💬【步驟 1 學習指引與 AI 提問 Prompt】

本專案要學會：階層式集群與 DBSCAN 密度分群。先檢查資料型態、缺失值、重複值與可疑欄位。

> ❓ 請用生活化比喻解釋這一步的原理，並指出一個常見誤解。

> 🛠️ 請只改一個超參數，比較修改前後的結果；不要使用測試集選參數。


---
### 【步驟 2】：選擇客群特徵與標準化
距離型演算法會受數值尺度影響；先將年齡、收入和消費指數改用同一把尺。


In [ ]:
# [步驟 2 程式碼] 特徵選擇與標準化
from sklearn.preprocessing import StandardScaler
features = ["年齡", "收入(千)", "消費指數(1-100)"]
X = df[features]
display(X.describe().T)
X_scaled = StandardScaler().fit_transform(X)


#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

收入與消費指數尺度不同；距離型演算法必須先用同一把尺。

> ❓ 請用生活化比喻解釋這一步的原理，並指出一個常見誤解。

> 🛠️ 請只改一個超參數，比較修改前後的結果；不要使用測試集選參數。


---
### 【步驟 3】：階層式集群與樹狀圖
從每位顧客各自成群開始逐步合併，利用樹狀圖觀察不同群數的可能性。


In [ ]:
# [步驟 3 程式碼] 階層式集群與樹狀圖
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.cluster import AgglomerativeClustering
linked = linkage(X_scaled, method="ward")
plt.figure(figsize=(14,5)); dendrogram(linked, truncate_mode="lastp", p=25)
plt.title("階層式集群樹狀圖"); plt.ylabel("合併距離"); plt.show()
df["階層式群組"] = AgglomerativeClustering(n_clusters=4).fit_predict(X_scaled)
display(df.groupby("階層式群組")[features].mean().round(1))


#### 💬【步驟 3 學習指引與 AI 提問 Prompt】

階層式集群由每人一群開始合併；樹狀圖切線的高度決定群數。

> ❓ 請用生活化比喻解釋這一步的原理，並指出一個常見誤解。

> 🛠️ 請只改一個超參數，比較修改前後的結果；不要使用測試集選參數。


---
### 【步驟 4】：DBSCAN 密度分群與雜訊
DBSCAN 不必先指定群數，但要用資料密度選擇 `eps` 與 `min_samples`。


In [ ]:
# [步驟 4 程式碼] DBSCAN 找密集客群與雜訊
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
distances, _ = NearestNeighbors(n_neighbors=5).fit(X_scaled).kneighbors(X_scaled)
plt.plot(np.sort(distances[:,-1])); plt.title("第 5 近鄰距離圖"); plt.show()
df["DBSCAN群組"] = DBSCAN(eps=.75, min_samples=5).fit_predict(X_scaled)
print("雜訊（-1）筆數：", (df["DBSCAN群組"] == -1).sum())
sns.scatterplot(data=df, x="收入(千)", y="消費指數(1-100)", hue="DBSCAN群組", palette="tab10", s=70)
plt.title("DBSCAN 分群結果"); plt.show()


#### 💬【步驟 4 學習指引與 AI 提問 Prompt】

DBSCAN 的 -1 是雜訊，不是第負一群；試著比較 eps 太小或太大時的結果。

> ❓ 請用生活化比喻解釋這一步的原理，並指出一個常見誤解。

> 🛠️ 請只改一個超參數，比較修改前後的結果；不要使用測試集選參數。


---
### 【步驟 5】：客群輪廓與方法限制
群組編號沒有大小意義；請用群內平均特徵與人數為每群命名、提出行銷策略。


In [ ]:
# [步驟 5 程式碼] 客群輪廓與方法限制
display(df.groupby("階層式群組")[features].agg(["mean","count"]).round(1))
print("階層式集群與 DBSCAN 沒有原生 predict；新客戶不能直接套用舊標籤。")


#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

群組編號沒有高低順序，請以各群平均輪廓與人數提出行銷建議。

> ❓ 請用生活化比喻解釋這一步的原理，並指出一個常見誤解。

> 🛠️ 請只改一個超參數，比較修改前後的結果；不要使用測試集選參數。


---
### 【步驟 6】：實務決策模擬
抽一位既有顧客，根據其所在群組提出一個可驗證的行銷假設。


In [ ]:
# [步驟 6 程式碼] 隨機抽取既有顧客的群組建議
case = df.sample(1, random_state=42)
display(case[features + ["階層式群組", "DBSCAN群組"]])
print("請先以收入、消費指數與年齡描述此顧客，再提出一項可 A/B 測試的行銷策略。")


#### 💬【步驟 6 學習指引與 AI 提問 Prompt】

分群是提出假設的工具，不是自動化的行銷決策。驗證策略時仍應設計對照組與成功指標。

> ❓ 請用生活化比喻解釋這一步的原理，並指出一個常見誤解。

> 🛠️ 請只改一個超參數，比較修改前後的結果；不要使用測試集選參數。
